In [1]:
"""
Constructs the meshes from the volumetric data and from the 2.5D shapes.
"""

import napari_spatialdata.constants.config
import spatialdata as sd
from pathlib import Path
from numpy.random import default_rng

from tissue_map_tools.igneous_converters import (  # noqa: F401
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.data_model.annotations_utils import (
    make_dtypes_compatible_with_precomputed_annotations,
)
import time  # noqa: F401
import shutil  # noqa: F401
from tissue_map_tools.converters import (  # noqa: F401
    from_spatialdata_points_to_precomputed_points,
)

RNG = default_rng(42)

/Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
out_path = Path.cwd() / "data"
sdata_zarr_path = out_path / "merfish_mouse_ileum.sdata.zarr"
precomputed_path = out_path / "merfish_mouse_ileum_precomputed"

# load the data
f = Path(sdata_zarr_path)
sdata = sd.read_zarr(f)

In [3]:
print(sd.get_extent(sdata["molecules"]))

{'x': (np.float64(112.0), np.float64(5720.0)), 'y': (np.float64(0.0), np.float64(9391.0)), 'z': (np.float64(0.0), np.float64(110.1455251))}


In [4]:
##
# subset the data
sdata_small = sd.bounding_box_query(
    sdata,
    axes=("x", "y", "z"),
    min_coordinate=[4000, 0, -10],
    max_coordinate=[5000, 1500, 200],
    target_coordinate_system="global",
)

/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: UserWarning: The object has `points` element. Depending on the number of points, querying MAY suffer from performance issues. Please consider filtering the object before calling this function by calling the `subset()` method of `SpatialData`.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
# we need to transform the vector data to match the image due to this issue:
# https://github.com/hms-dbmi/tissue-map-tools/issues/13
transformation = sd.transformations.get_transformation(sdata_small["stains"])
translation_vector = transformation.to_affine_matrix(
    input_axes=("x", "y", "z"), output_axes=("x", "y", "z")
)[:3, 3]
translation = sd.transformations.Translation(translation_vector, axes=("x", "y", "z"))
for _, element_name, _ in sdata_small.gen_spatial_elements():
    old_transformation = sd.transformations.get_transformation(
        sdata_small[element_name]
    )
    sequence = sd.transformations.Sequence([old_transformation, translation.inverse()])
    sd.transformations.set_transformation(
        sdata_small[element_name],
        transformation=sequence,
        to_coordinate_system="global",
    )
    if sd.models.get_model(sdata_small[element_name]) not in (
        sd.models.Image3DModel,
        sd.models.Labels3DModel,
    ):
        transformed = sd.transform(sdata_small[element_name], to_coordinate_system="global")
        sdata_small[element_name] = transformed

sdata = sdata_small

In [6]:
##
#
# cells_baysor_cropped = sd.bounding_box_query(
#     sdata["cells_baysor"],
#     axes=("x", "y", "z"),
#     min_coordinate=[1500, 1500, -10],
#     max_coordinate=[3000, 3000, 200],
#     target_coordinate_system="global",
# )
# sdata["cells_baysor"] = cells_baysor_cropped

##
from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
    raster=sdata["dapi_labels"],
    precomputed_path=str(precomputed_path),
)

##
# from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
#     raster=sdata["membrane_labels"],
#     precomputed_path=str(precomputed_path),
# )


##

Converted OME-Zarr data to the Precomputed format (segmentation) at /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/merfish_mouse_ileum_precomputed with pixel sizes {'x': 1000, 'y': 1000, 'z': 13768} and axes ['x', 'y', 'z'].
Volume Bounds:  Bbox([0, 0, 0],[1000, 1500, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[1000, 1500, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.06it/s]


Volume Bounds:  Bbox([0, 0, 0],[500, 750, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[500, 750, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 15.15it/s]


Volume Bounds:  Bbox([0, 0, 0],[250, 375, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[250, 375, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 36.92it/s]


Volume Bounds:  Bbox([0, 0, 0],[125, 188, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[125, 188, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.84it/s]


In [7]:
subset = RNG.choice(len(sdata["molecule_baysor"]), 10000, replace=False)

print(sdata["molecule_baysor"].columns)
subset_df = sdata["molecule_baysor"].compute().iloc[subset]
# subset_df = sdata["molecule_baysor"].compute()
subset_df = subset_df[
    [
        # working
        "x",
        "y",
        "z",
        "gene",
        "area",
        "mol_id",
        "x_raw",
        "y_raw",
        "z_raw",
        "brightness",
        "total_magnitude",
        "compartment",
        "nuclei_probs",
        "assignment_confidence",
        #
        "cell",
        "is_noise",  # TODO: bool not working at the moment
        # "ncv_color",  # TODO: represent as RGB
        "layer",
    ]
]
subset_df

Index(['mol_id', 'x_raw', 'y_raw', 'z_raw', 'gene', 'area', 'brightness',
       'total_magnitude', 'qc_score', 'molecule_id', 'confidence',
       'compartment', 'nuclei_probs', 'cell', 'assignment_confidence',
       'is_noise', 'ncv_color', 'layer', 'x', 'y', 'z'],
      dtype='object')


,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
460642,500.0,841.0,2.753638e+01,Lpar1,15,10095018,-2630.866,-1265.363,5.5,1.875414,1125.9140,Cyto,0.025976,0.675,867,False,3
423082,952.0,761.0,5.507276e+01,Clca3b,6,10003055,-2581.609,-1274.128,8.5,2.150356,848.2183,Unknown,0.986717,1.000,1045,False,5
413076,84.0,444.0,-2.945800e-09,Txndc5,4,9988714,-2676.138,-1308.672,2.5,2.008227,407.6496,Unknown,0.967683,1.000,511,False,1
444843,715.0,1069.0,4.130457e+01,Gp2,3,10048842,-2607.459,-1240.554,7.0,2.013750,309.6504,Unknown,0.940632,0.825,1085,False,4
451885,292.0,1144.0,-2.945800e-09,Sdc1,4,10069068,-2653.550,-1232.399,2.5,2.038763,437.3442,Unknown,0.818268,0.825,0,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426314,356.0,1094.0,-2.945800e-09,Adgrf5,15,10011807,-2646.565,-1237.818,2.5,2.164938,2192.9530,Unknown,1.000000,0.875,5284,False,1
426622,862.0,827.0,6.884095e+01,Adgrf5,5,10012460,-2591.478,-1266.878,10.0,1.544077,175.0038,Cyto,0.019236,0.275,5702,False,6
442791,867.0,800.0,4.130457e+01,Nlrp6,4,10044869,-2590.909,-1269.846,7.0,1.915653,329.3920,Cyto,0.091038,0.950,1022,False,4
450905,183.0,1297.0,2.753638e+01,Mzb1,4,10067050,-2665.387,-1215.732,5.5,1.781510,241.8634,Unknown,1.000000,1.000,905,False,3


In [8]:
make_dtypes_compatible_with_precomputed_annotations(
    subset_df,
    max_categories=250,
    check_for_overflow=True,
)

,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
460642,500.0,841.0,2.753638e+01,Lpar1,15,10095018,-2630.865967,-1265.363037,5.5,1.875414,1125.913940,Cyto,0.025976,0.675,867,0,3
423082,952.0,761.0,5.507276e+01,Clca3b,6,10003055,-2581.608887,-1274.128052,8.5,2.150356,848.218323,Unknown,0.986717,1.000,1045,0,5
413076,84.0,444.0,-2.945800e-09,Txndc5,4,9988714,-2676.137939,-1308.671997,2.5,2.008227,407.649597,Unknown,0.967683,1.000,511,0,1
444843,715.0,1069.0,4.130457e+01,Gp2,3,10048842,-2607.458984,-1240.553955,7.0,2.013750,309.650391,Unknown,0.940632,0.825,1085,0,4
451885,292.0,1144.0,-2.945800e-09,Sdc1,4,10069068,-2653.550049,-1232.399048,2.5,2.038763,437.344208,Unknown,0.818268,0.825,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426314,356.0,1094.0,-2.945800e-09,Adgrf5,15,10011807,-2646.564941,-1237.817993,2.5,2.164938,2192.952881,Unknown,1.000000,0.875,5284,0,1
426622,862.0,827.0,6.884095e+01,Adgrf5,5,10012460,-2591.478027,-1266.878052,10.0,1.544078,175.003799,Cyto,0.019236,0.275,5702,0,6
442791,867.0,800.0,4.130457e+01,Nlrp6,4,10044869,-2590.908936,-1269.845947,7.0,1.915653,329.391998,Cyto,0.091038,0.950,1022,0,4
450905,183.0,1297.0,2.753638e+01,Mzb1,4,10067050,-2665.386963,-1215.732056,5.5,1.781510,241.863403,Unknown,1.000000,1.000,905,0,3


In [9]:
sdata["molecule_baysor"] = sd.models.PointsModel.parse(subset_df)

# TODO: temporary workaround: raster data converted to precomputed expresses units in nm
#  therefore let's multiply the points by 1000
for ax in ["x", "y", "z"]:
    sdata["molecule_baysor"][ax] = sdata["molecule_baysor"][ax] * 1000


##
# debug
points = sdata["molecule_baysor"].compute().iloc[:2]
print("point 0")
print(points.iloc[0])
print("")
print("point 1")
print(points.iloc[1])
print("")
print(points.x.dtype)
# print(points.gene.cat.categories)
print(points.gene.cat.categories.get_loc(points.gene.iloc[0]))
##
print("converting the points to the precomputed format")

# TODO: there should be no need to add the subpath (we should be able to specify the
#  parent cloud volume object
# TODO: the info file in the parent volume should be updated to include the points
# TODO: the view APIs show include the points

start = time.time()
path = precomputed_path / "molecule_baysor"
if path.exists():
    shutil.rmtree(path)

/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:946: UserWarning: The index of the dataframe is not monotonic increasing. It is recommended to sort the data to adjust the order of the index before calling .parse() (or call `parse(sort=True)`) to avoid possible problems due to unknown divisions.
  return method.__get__(obj, cls)(*args, **kwargs)


point 0
x                            500000.0
y                            841000.0
z                        27536.380859
gene                            Lpar1
area                               15
mol_id                       10095018
x_raw                    -2630.865967
y_raw                    -1265.363037
z_raw                             5.5
brightness                   1.875414
total_magnitude            1125.91394
compartment                      Cyto
nuclei_probs                 0.025976
assignment_confidence           0.675
cell                              867
is_noise                            0
layer                               3
Name: 460642, dtype: object

point 1
x                            952000.0
y                            761000.0
z                        55072.761719
gene                           Clca3b
area                                6
mol_id                       10003055
x_raw                    -2581.608887
y_raw                    -1274.128052
z_raw

In [10]:
from_spatialdata_points_to_precomputed_points(
    sdata["molecule_baysor"],
    precomputed_path=precomputed_path,
    points_name="molecule_baysor",
    limit=1000,
    # limit=500,
)
print(f"conversion of points: {time.time() - start}")

Processing grid level 0 with shape (1, 1, 1) and chunk size [ 998000.        1498000.         110145.5234375]. Remaining points: 10000
Emitting 1000 points for grid cell (0, 0, 0)
Processing grid level 1 with shape (2, 2, 1) and chunk size [499000.        749000.        110145.5234375]. Remaining points: 9000
Emitting 1000 points for grid cell (0, 0, 0)
Emitting 1000 points for grid cell (0, 1, 0)
Emitting 1000 points for grid cell (1, 0, 0)
Emitting 1000 points for grid cell (1, 1, 0)
Processing grid level 2 with shape (4, 4, 1) and chunk size [249500.        374500.        110145.5234375]. Remaining points: 5000
Emitting 438 points for grid cell (0, 0, 0)
Emitting 502 points for grid cell (0, 1, 0)
Emitting 178 points for grid cell (1, 0, 0)
Emitting 239 points for grid cell (1, 1, 0)
Emitting 530 points for grid cell (0, 2, 0)
Emitting 357 points for grid cell (0, 3, 0)
Emitting 318 points for grid cell (1, 2, 0)
Emitting 263 points for grid cell (1, 3, 0)
Emitting 265 points for gr

In [11]:
print('done')

done
